# ABS Population

Unified notebook combining monthly arrivals and departures (ABS 3401.0), the quarterly estimated resident population (ABS 3101.0), and multi-source population growth measures (ABS 3101.0, 3401.0, 6202.0, 5206.0).

Supersedes: *ABS Monthly Arrivals Departures 3401*, *ABS Quarterly Estimated Resident Population 3101*, *ABS Population Growth Measures*.

## Python set-up

In [1]:
# system imports
import re
import textwrap
from functools import cache
from typing import cast

In [2]:
# analytic imports
import pandas as pd
from pandas import PeriodIndex, Series

import readabs as ra
from readabs import metacol as mc
from readabs import recalibrate

from statsmodels.tsa.filters.hp_filter import hpfilter

from mgplot import (
    abbreviate_state,
    clear_chart_dir,
    finalise_plot,
    get_color,
    growth_plot_finalise,
    line_plot_finalise,
    multi_start,
    postcovid_plot_finalise,
    revision_plot_finalise,
    seastrend_plot_finalise,
    series_growth_plot_finalise,
    set_chart_dir,
)

In [3]:
# local imports
import decompose
# abs_population is the single source of truth for population *levels*:
#   get_population("civ15", ...)  -> 6202.0 civilian pop 15+
#   get_population("implicit")    -> 5206.0 GDP-implied population
# ERP is fetched directly from 3101.0 table 310101 below (not via the module)
# because that batch also needs ERP's siblings - annual growth/rate, natural
# increase, births, deaths - which live only in 310101 and the module does
# not expose. (The module's ERP comes from 310104 in persons, vs 310101 in '000.)
from abs_population import (
    get_population, smoothed_monthly_pop_growth, _erp_age_sum, _interp_june,
)
from abs_structured_capture import ReqsDict, ReqsTuple, get_abs_data, to_scale_word
from henderson import hma

In [4]:
# pandas display settings
pd.options.display.max_rows = 999999
pd.options.display.max_columns = 999
pd.options.display.max_colwidth = 999

# --- notebook constants ---
SHOW = False
FILE_TYPE = "png"
CHART_DIR = "./CHARTS/Population/"

# Window sizes for multi_start pairings
RECENT_MONTHS = 88              # ~7 years, used for monthly growth charts
QUARTERLY_WINDOW = (0, -15)     # quarters, used for ERP quarterly bar/line charts

# Source attributions
SOURCE_3101 = "ABS: 3101.0"
SOURCE_3401 = "ABS: 3401.0"
SOURCE_POP = "ABS Cat. 3101.0, 5206.0, 6202.0"  # for population-estimate plots

# Age-profile constants
AGE_TABLES = tuple("31010" + str(i) for i in range(51, 60))
GROUPS = ("Female", "Male", "Persons")  # Persons must be last
STATES = {"NSW", "Vic", "Qld", "SA", "WA", "Tas", "NT", "ACT", "Australia"}
STATE_COLORS = {state: get_color(state) for state in STATES}
HMA_SMOOTHER = 5
LINESTYLE = {"style": ["-", "-.", "--", ":"] * 3}

# COVID break-points by series (for quarterly seasonal decomposition)
DISCONTINUITIES = {
    "Births": [pd.Period("2020-Q4", freq="Q")],
    "Deaths": [],
    "Natural Increase": [pd.Period("2020-Q4", freq="Q")],
    "Overseas Arrivals": [pd.Period("2020-Q1", freq="Q")],
    "Overseas Departures": [pd.Period("2020-Q1", freq="Q")],
    "Net Overseas Migration": [pd.Period("2020-Q1", freq="Q")],
}

# Chart directory - set once for the whole notebook
set_chart_dir(CHART_DIR)
clear_chart_dir()

## Data capture

Each ABS table is fetched once via `readabs.read_abs_cat(single_excel_only=...)` (fast, cached). Growth-measure series from multiple catalogues are captured via `abs_structured_capture`.

In [5]:
# --- Primary ABS tables ---
# Quarterly aggregates: ERP, births, deaths, natural increase, migration
_d, meta_310101 = ra.read_abs_cat("3101.0", single_excel_only="310101", verbose=False)
df_310101 = _d["310101"]

# Quarterly ERP by state and territory
_d, meta_310104 = ra.read_abs_cat("3101.0", single_excel_only="310104", verbose=False)
df_310104 = _d["310104"]

# Monthly overseas arrivals and departures
_d, meta_340101 = ra.read_abs_cat("3401.0", single_excel_only="340101", verbose=False)
df_340101 = _d["340101"]

_d, meta_340102 = ra.read_abs_cat("3401.0", single_excel_only="340102", verbose=False)
df_340102 = _d["340102"]

# Combined arrivals/departures dictionary (used by 3401 plotting functions)
abs_3401 = {"340101": df_340101, "340102": df_340102}
meta_3401 = pd.concat([meta_340101, meta_340102]).drop_duplicates(subset=[mc.id])

# RECENT_M marks the START of the "recent history" window (not the latest date).
# Mirrors the hard-coded 2020-12 start used by abs_helper.get_abs_data.
RECENT_M = pd.Period("2020-12", freq="M")
PLOT_TIMES_M = (0, RECENT_M)

print(f"Arrivals/Departures current to: {df_340101.index[-1]}")
print(f"ERP (310101)         current to: {df_310101.index[-1]}")
print(f"State ERP (310104)   current to: {df_310104.index[-1]}")

Arrivals/Departures current to: 2026-04
ERP (310101)         current to: 2025Q4
State ERP (310104)   current to: 2025Q4


In [6]:
# --- Growth-measure series (multi-catalogue, cached) ---
@cache
def collect_growth_data() -> dict[str, Series]:
    """Fetch growth-measure series spanning 3101.0, 3401.0, 6202.0 and 5206.0
    and derive annualised aggregates. All quarterly series are upsampled to
    monthly. ABS-published unit strings are preserved on each raw Series via
    `.attrs['unit']` (e.g. '000', 'Percent'); derived series have no unit attr.
    Cached: the heavy lifting runs once per kernel."""

    sought_after: ReqsDict = {
        # ReqsTuple: cat, table, did, stype, unit, seek_yr_growth, calc_growth, zip_file
        "Estimated Resident Population": ReqsTuple(
            "3101.0", "310101",
            "Estimated Resident Population (ERP) ;  Australia ;",
            "O", "", False, False, "",
        ),
        "Estimated Resident Population Annual Growth": ReqsTuple(
            "3101.0", "310101",
            "ERP Change Over Previous Year ;  Australia ;",
            "O", "000", False, False, "",
        ),
        "Estimated Resident Population Annual Growth Rate": ReqsTuple(
            "3101.0", "310101",
            "Percentage ERP Change Over Previous Year ;  Australia ;",
            "O", "", False, False, "",
        ),
        "Natural Increase": ReqsTuple(
            "3101.0", "310101", "Natural Increase ;  Australia ;",
            "O", "", False, False, "",
        ),
        "Deaths": ReqsTuple(
            "3101.0", "310101", "Deaths ;  Australia ;",
            "O", "", False, False, "",
        ),
        "Births": ReqsTuple(
            "3101.0", "310101", "Births ;  Australia ;",
            "O", "", False, False, "",
        ),
        "Total Monthly Arrivals": ReqsTuple(
            "3401.0", "340101",
            "Number of movements ;  Total Arrivals ;",
            "O", "", False, False, "",
        ),
        "Total Monthly Departures": ReqsTuple(
            "3401.0", "340102",
            "Number of movements ;  Total Departures ;",
            "O", "", False, False, "",
        ),
    }

    # Fetch as AbsSeries records, then unwrap to Series carrying ABS unit on .attrs
    raw = get_abs_data(sought_after)
    data: dict[str, Series] = {}
    for key, record in raw.items():
        s = record.series.copy()
        s.attrs["unit"] = record.unit
        s.attrs["freq"] = record.freq
        data[key] = s

    # Civilian population 15+ (6202.0) sourced from abs_population (single source
    # of truth); returns the monthly '000 level, same as the old batch fetch.
    data["LFS Civilian Population 15+"] = get_population("civ15", state="Australia")[0]

    # --- derived series ---
    # Smoothed 6202 civilian-population-15+ growth. The raw level is benchmark-
    # interpolated (its month-on-month change is step-shaped), so smooth it into
    # a clean monthly increment and express both annual measures off that:
    # annual growth is the 12-month rolling sum of the increment; the annual rate
    # divides that sum by the level. Replaces the raw diff(12) and the ABS
    # calc_growth rate, which both carried the benchmark steps as kinks.
    _civpop_growth = smoothed_monthly_pop_growth(data["LFS Civilian Population 15+"])
    data["LFS Civilian Population 15+ Annual Growth"] = (
        _civpop_growth.rolling(12).sum()
    )
    data["LFS Civilian Population 15+ Annual Growth Rate"] = (
        _civpop_growth.rolling(12).sum() / data["LFS Civilian Population 15+"] * 100
    )
    data["12 month rolling net total arrivals"] = (
        (data["Total Monthly Arrivals"] - data["Total Monthly Departures"])
        .rolling(12).sum() / 1_000
    )
    data["Annual Natural Increase"] = data["Natural Increase"].rolling(4).sum()
    data["Annual Deaths"] = data["Deaths"].rolling(4).sum()
    data["Annual Births"] = data["Births"].rolling(4).sum()
    data["ERP Growth less Natural Increase"] = (
        data["Estimated Resident Population Annual Growth"]
        - data["Annual Natural Increase"]
    )
    # Implicit population from the National Accounts (5206.0), via abs_population.
    # get_population("implicit") returns persons; rescale to '000 to match the
    # other population series in this dict.
    data["Implicit population from National Accounts"] = (
        get_population("implicit")[0] / 1_000
    )
    data["Implicit population (from National Accounts) growth"] = (
        data["Implicit population from National Accounts"].diff(periods=4)
    )
    data["Implicit population (from National Accounts) growth rate"] = (
        data["Implicit population from National Accounts"].pct_change(
            periods=4, fill_method=None,
        ) * 100
    )

    # --- upsample quarterly series to monthly for consistent plotting ---
    new_dict = {}
    for key, series in data.items():
        periodicity = cast(PeriodIndex, series.index).freqstr[0]
        if periodicity == "Q":
            series = ra.qtly_to_monthly(series)
        new_dict[key] = series
    return new_dict

## Arrivals and Departures (3401.0)

Monthly arrivals and departures, by category. The Total Overseas 12-month net flow is omitted here because it is plotted with HP-filter and Henderson smoothing in the Population Growth & Migration section below.

In [7]:
def _plt_additional_charts(title: str, stype: str, units: str, plt_df: pd.DataFrame) -> None:
    """For Permanent and Long-term: net monthly flows + seasonal decomposition."""

    period = 12
    net_monthly = plt_df.iloc[:, 0] - plt_df.iloc[:, 1]
    net_monthly, net_monthly_units = recalibrate(net_monthly, units)
    rolling = net_monthly.rolling(period, min_periods=period).mean()
    dataset = pd.DataFrame({
        "Net Monthly Arrivals-Departures": net_monthly,
        "12m Rolling Mean": rolling,
    })
    multi_start(
        dataset,
        function=line_plot_finalise,
        starts=PLOT_TIMES_M,
        title=f"{title}: Net Monthly Arrivals-Departures",
        ylabel=f"{net_monthly_units} / month",
        rfooter=SOURCE_3401,
        lfooter=f"Australia. {stype} series. ",
        width=[1, 2],
        annotate=True,
        y0=True,
        pre_tag="arr",
        show=SHOW,
    )

    selector = "Seasonally Adjusted"
    selected = {}
    for column, series in plt_df.items():
        decomposed = decompose.decompose(
            series.dropna(),
            arima_extend=True,
            ignore_years=(2020, 2021),  # COVID
        )[["Seasonally Adjusted", "Trend"]]
        selected[column] = decomposed[selector]
        multi_start(
            decomposed,
            function=seastrend_plot_finalise,
            starts=PLOT_TIMES_M,
            title=f"{column} - Seasonal Decomposition",
            ylabel=f"{units} / month",
            rfooter=SOURCE_3401,
            lfooter="Australia. ",
            y0=True,
            pre_tag="arr",
            show=SHOW,
        )

    plot_trends = pd.DataFrame(selected)
    multi_start(
        plot_trends,
        function=line_plot_finalise,
        starts=PLOT_TIMES_M,
        title=f"{title} movements - {selector}",
        ylabel=f"{units} / month",
        rfooter=SOURCE_3401,
        annotate=True,
        lfooter=f"Australia. Inhouse seasonal decomposition. {selector} series. ",
        y0=True,
        pre_tag="arr",
        show=SHOW,
    )


def plot_headline_pairs() -> None:
    """Headline arrival/departure paired charts, by category."""

    pairs = {
        "Total Overseas": (
            ("Number of movements ;  Total Arrivals ;", "340101"),
            ("Number of movements ;  Total Departures ;", "340102"),
        ),
        "Short-term residents": (
            ("Number of movements ;  Short-term Residents returning ;", "340101"),
            ("Number of movements ;  Short-term Residents departing ;", "340102"),
        ),
        "Short-term visitors": (
            ("Number of movements ;  Short-term Visitors arriving ;", "340101"),
            ("Number of movements ;  Short-term Visitors departing ;", "340102"),
        ),
        "Permanent and Long-term": (
            ("Number of movements ;  Permanent and Long-term Arrivals ;", "340101"),
            ("Number of movements ;  Permanent and Long-term Departures ;", "340102"),
        ),
    }

    stype = "Original"
    for title, plotable in pairs.items():
        plt_df = pd.DataFrame()
        units = ""
        for did, table in plotable:
            search = {did: mc.did, table: mc.table, stype: mc.stype}
            _t, series_id, units = ra.find_abs_id(meta_3401, search)
            plt_df[did.split(";")[-2].strip()] = abs_3401[table][series_id]
        plt_df, units = recalibrate(plt_df, units)

        # paired arrivals-and-departures line chart
        multi_start(
            plt_df,
            function=line_plot_finalise,
            starts=PLOT_TIMES_M,
            title=f"{title}: Arrivals and Departures",
            ylabel=f"{units} / month",
            rfooter=SOURCE_3401,
            lfooter=f"Australia. {stype} series. ",
            legend=True,
            pre_tag="arr",
            show=SHOW,
        )

        # 12m rolling net flow (single line, no smoothing)
        annual = plt_df.rolling(12, min_periods=12).sum()
        net_annual = annual.iloc[:, 0] - annual.iloc[:, 1]
        net_annual, net_annual_units = recalibrate(net_annual, units)
        multi_start(
            net_annual,
            function=line_plot_finalise,
            starts=PLOT_TIMES_M,
            title=f"{title}: Arrivals-Departures (12m rolling sum)",
            ylabel=f"{net_annual_units} / year",
            rfooter=SOURCE_3401,
            lfooter=f"Australia. {stype} series. ",
            annotate=True,
            y0=True,
            pre_tag="arr",
            show=SHOW,
        )

        if title == "Permanent and Long-term":
            _plt_additional_charts(title, stype, units, plt_df)

plot_headline_pairs()

In [8]:
def plot_individual_movements() -> None:
    """Line chart + postcovid chart for each individual arrivals/departures series."""

    tables = ("340101", "340102")
    arrivals = meta_3401.loc[meta_3401[mc.table] == tables[0], mc.did]
    departures = meta_3401.loc[meta_3401[mc.table] == tables[1], mc.did]

    stype = "Original"
    for movement, table in zip((arrivals, departures), tables):
        data = abs_3401[table]
        for did in movement:
            search = {did: mc.did, table: mc.table, stype: mc.stype}
            _t, series_id, units = ra.find_abs_id(
                meta_3401, search, exact_match=True, verbose=False,
            )
            series, units = recalibrate(data[series_id], units)
            chart_title = did.split(";")[-2]

            multi_start(
                series,
                starts=PLOT_TIMES_M,
                function=line_plot_finalise,
                title=chart_title,
                ylabel=f"{units} / month",
                rfooter=SOURCE_3401,
                lfooter=f"Australia. {stype} series. ",
                annotate=True,
                y0=True,
                pre_tag="arr",
                show=SHOW,
            )
            postcovid_plot_finalise(
                series,
                title=chart_title,
                ylabel=f"{units} / month",
                rfooter=SOURCE_3401,
                lfooter=f"Australia. {stype} series. ",
                y0=True,
                pre_tag="arr",
                show=SHOW,
            )


plot_individual_movements()

In [9]:
def plot_net_migration_hp() -> None:
    """12m net arrivals with HP-filter trend (filter applied separately around COVID)."""

    pop = collect_growth_data()
    net_arrivals = pop["12 month rolling net total arrivals"].dropna()

    covid_start = pd.Period("2020-03", freq="M")
    covid_end = pd.Period("2024-09", freq="M")
    pre_covid = net_arrivals[net_arrivals.index < covid_start]
    post_covid = net_arrivals[net_arrivals.index > covid_end]

    _, pre_trend = hpfilter(pre_covid, lamb=129600)
    _, post_trend = hpfilter(post_covid, lamb=129600)
    hp_trend = pd.concat([
        pd.Series(pre_trend, index=pre_covid.index),
        pd.Series(post_trend, index=post_covid.index),
    ])

    frame = pd.DataFrame({
        "12m Rolling Net Arrivals": net_arrivals,
        "HP Filter Trend": hp_trend,
    })
    frame, units = recalibrate(frame, "Thousands")

    multi_start(
        frame,
        function=line_plot_finalise,
        starts=(0, -RECENT_MONTHS),
        title="Net Migration Proxy: 12m Rolling Net Arrivals (HP Filtered)",
        ylabel=f"{units} / year",
        dropna=True,
        width=[1, 2],
        y0=True,
        annotate=True,
        legend=True,
        lfooter=(
            "Australia. Original series. "
            "Net arrivals: total arrivals minus total departures. "
            "HP filter excludes Mar 2020 to Sep 2024. "
        ),
        rfooter=SOURCE_3401,
        pre_tag="arr",
        show=SHOW,
        file_type=FILE_TYPE,
    )


def plot_net_migration_hma() -> None:
    """12m net arrivals with 25-term Henderson moving average trend."""

    pop = collect_growth_data()
    net_arrivals = pop["12 month rolling net total arrivals"].dropna()
    hma_trend = hma(net_arrivals, 25)

    frame = pd.DataFrame({
        "12m Rolling Net Arrivals": net_arrivals,
        "25-term HMA Trend": hma_trend,
    })
    frame, units = recalibrate(frame, "Thousands")

    multi_start(
        frame,
        function=line_plot_finalise,
        starts=(0, -RECENT_MONTHS),
        title="Net Migration Proxy: 12m Rolling Net Arrivals (HMA Smoothed)",
        ylabel=f"{units} / year",
        dropna=True,
        width=[1, 2],
        y0=True,
        annotate=True,
        legend=True,
        lfooter=(
            "Australia. Original series. "
            "Net arrivals: total arrivals minus total departures. "
            "25-term Henderson moving average. "
        ),
        rfooter=SOURCE_3401,
        pre_tag="arr",
        show=SHOW,
        file_type=FILE_TYPE,
    )


plot_net_migration_hp()
plot_net_migration_hma()

## Estimated Resident Population (3101.0)

### Data revisions

Tracks how headline series have changed across the last six ABS prints.

In [10]:
def plot_data_revisions() -> None:
    """Plot revisions in headline 310101 series across the last six ABS prints."""

    how_far_back = 6
    dataset = [
        "Estimated Resident Population",
        "Births ;  Australia ;",
        "Deaths ;  Australia ;",
        "Natural Increase ;  Australia ;",
        "Overseas Arrivals ;  Australia ;",
        "Overseas Departures ;  Australia ;",
        "Net Overseas Migration ;  Australia ;",
    ]
    stype = "Original"
    for series in dataset:
        u = None
        repository = pd.DataFrame()
        history = None
        for _i in range(how_far_back):
            d, m = ra.read_abs_cat(
                "3101.0", single_excel_only="310101", history=history,
            )
            selector = {series: mc.did, stype: mc.stype}
            t, s, u = ra.find_abs_id(m, selector, regex=False, verbose=False)
            date = f"ABS print for {d[t].index[-1].strftime('%Y-%b')}"
            repository[date] = d[t][s]
            history = (d[t].index[-1] - 1).strftime("%b-%Y").lower()

        if u is None:
            continue

        revision_plot_finalise(
            data=repository,
            ylabel=u,
            title=f"Data revisions: {re.sub(':.*$', '', series)}",
            rfooter=SOURCE_3101,
            lfooter=f"Australia. {stype}. ",
            legend={"loc": "best", "fontsize": 9},
            pre_tag="erp",
            show=SHOW,
        )
        if series == "Estimated Resident Population":
            revision_plot_finalise(
                data=repository.diff(1),
                ylabel=u,
                title=f"Data revisions: {re.sub(':.*$', '', series)} Growth",
                rfooter=SOURCE_3101,
                lfooter=f"Australia. {stype}. ",
                legend={"loc": "best", "fontsize": 9},
                pre_tag="erp",
                show=SHOW,
            )


plot_data_revisions()

### Key demographic series

In [11]:
def plot_key_demographics() -> None:
    """Births, deaths, natural increase, arrivals, departures, net migration.
    Raw series + in-house seasonal decomposition + postcovid chart."""

    series_type = "Original"
    key_list = [
        "Births",
        "Deaths",
        "Natural Increase",
        "Overseas Arrivals",
        "Overseas Departures",
        "Net Overseas Migration",
    ]

    for chart in key_list:
        selector = {"310101": mc.table, series_type: mc.stype, chart: mc.did}
        _t, sid, units = ra.find_abs_id(meta_310101, selector, verbose=False)
        series = df_310101[sid]
        series.name = chart
        series, units = recalibrate(series, units)

        common = {
            "title": chart,
            "y0": True,
            "ylabel": f"{units} / Quarter",
            "rfooter": SOURCE_3101,
            "pre_tag": "erp",
            "show": SHOW,
        }
        line_plot_finalise(
            series,
            lfooter=f"Australia. {series_type} series. ",
            **common,
        )

        common["starts"] = QUARTERLY_WINDOW
        decomposed = decompose.decompose(
            series.dropna(),
            constant_seasonal=True,
            arima_extend=True,
            discontinuity_list=DISCONTINUITIES[chart],
            ignore_years=(2020, 2021),
        )
        multi_start(
            decomposed[["Seasonally Adjusted", "Trend"]],
            function=seastrend_plot_finalise,
            tag="sa-trend",
            lfooter="Australia. Seasonally adjusted using in-house methods. ",
            **common,
        )
        postcovid_common = {k: v for k, v in common.items() if k != "starts"}
        postcovid_plot_finalise(
            decomposed["Seasonally Adjusted"],
            tag="covid-recovery",
            lfooter=(
                "Australia. Seasonally adjusted series plotted. "
                "Seasonally adjusted using in-house methods. "
            ),
            **postcovid_common,
        )


plot_key_demographics()

### Natural increase ratios

In [12]:
def plot_natural_increase_ratios() -> None:
    """Natural increase as % of population, and as % of gross increase."""

    series_type = "Original"
    series_map = {}
    for name in ("Estimated Resident Population", "Natural Increase", "Deaths"):
        selector = {"310101": mc.table, series_type: mc.stype, name: mc.did}
        _t, sid, _u = ra.find_abs_id(meta_310101, selector, verbose=False)
        series_map[name] = df_310101[sid]

    erp = series_map["Estimated Resident Population"]
    natural_increase = series_map["Natural Increase"]
    deaths = series_map["Deaths"]

    ni_pct_pop = (natural_increase.rolling(4).sum() / erp) * 100
    ni_pct_pop.name = "Natural Increase (% of population, annualised)"
    multi_start(
        ni_pct_pop.dropna(),
        function=line_plot_finalise,
        starts=QUARTERLY_WINDOW,
        title="Natural Increase as Percentage of Population",
        ylabel="Per cent per year",
        y0=True,
        lfooter=f"Australia. {series_type} series. Annualised. ",
        rfooter=f"{SOURCE_3101} 310101",
        annotate=True,
        pre_tag="erp",
        show=SHOW,
    )

    gross_increase = erp.diff() + deaths
    ni_pct_gross = (natural_increase / gross_increase) * 100
    ni_pct_gross.name = "Natural Increase (% of gross increase)"
    multi_start(
        ni_pct_gross.dropna(),
        function=line_plot_finalise,
        starts=QUARTERLY_WINDOW,
        title="Natural Increase as Percentage of Gross Population Increase",
        ylabel="Per cent",
        y0=True,
        lfooter=(
            f"Australia. {series_type} series. Gross increase = ERP change + deaths. "
        ),
        rfooter=f"{SOURCE_3101} 310101",
        annotate=True,
        pre_tag="erp",
        show=SHOW,
    )


plot_natural_increase_ratios()

### National and state populations

In [13]:
def plot_state_erp() -> None:
    """ERP by state/territory: raw, postcovid, and growth variants."""

    erp_phrase = "Estimated Resident Population ;  Persons ;"
    states = (
        meta_310104.loc[
            (meta_310104[mc.table] == "310104")
            & (meta_310104[mc.did].str.contains(erp_phrase)),
            mc.did,
        ]
        .str.replace(erp_phrase, "")
        .str.replace(" ;", "")
        .str.strip()
        .to_list()
    )

    for state in states:
        selector = {
            "310104": mc.table,
            erp_phrase: mc.did,
            f";  {state} ;": mc.did,
        }
        _t, sid, units = ra.find_abs_id(meta_310104, selector, verbose=False)
        series = df_310104[sid]
        series.name = "Estimated Resident Population"
        units = "Number Persons" if units == "Persons" else units
        series, units = recalibrate(series, units)

        title = f"Estimated Resident Population: {abbreviate_state(state)}"
        line_plot_finalise(
            series,
            title=title,
            ylabel=units,
            rfooter=f"{SOURCE_3101} 310104",
            width=2,
            pre_tag="erp",
            show=SHOW,
        )
        postcovid_plot_finalise(
            series,
            title=title,
            ylabel=units,
            tag="-covid",
            rfooter=f"{SOURCE_3101} 310104",
            pre_tag="erp",
            show=SHOW,
        )
        for start in (0, -13):
            series_growth_plot_finalise(
                series,
                plot_from=start,
                tag=f"growth-{start}",
                title=f"Growth in the {title}",
                rfooter=f"{SOURCE_3101} 310104",
                pre_tag="erp",
                show=SHOW,
            )


plot_state_erp()

### Quarterly and through-the-year growth in persons

Absolute population increments (not per cent) for the Estimated Resident Population and Net Overseas Migration, Australia.

In [14]:
def plot_level_growth() -> None:
    """ERP and NOM: quarterly increment (bars) and through-the-year total (line),
    in persons - recent period only. Uses growth_plot_finalise (the 'level growth'
    pattern from the 6202 notebook), which labels the bars 'Quarterly Growth' and
    the line 'Annual Growth' (both increments in persons, not per cent). ERP is a
    stock, so its increments are first differences; NOM is already a quarterly
    flow, so its quarterly value is the increment and its 4-quarter rolling sum is
    the annual total."""

    stype = "Original"
    recent = QUARTERLY_WINDOW[1]

    configs = (
        ("Estimated Resident Population: Quarterly and Annual Growth",
         "Estimated Resident Population (ERP) ;  Australia ;", "diff"),
        ("Net Overseas Migration: Quarterly and Annual Totals",
         "Net Overseas Migration ;  Australia ;", "flow"),
    )
    for title, did, kind in configs:
        selector = {"310101": mc.table, stype: mc.stype, did: mc.did}
        _t, sid, _u = ra.find_abs_id(meta_310101, selector, verbose=False)
        level = df_310101[sid]
        if kind == "diff":  # stock: increment is the first difference
            annual, periodic = level.diff(4), level.diff(1)
        else:  # flow: quarterly value is the increment; annual = 4-quarter sum
            annual, periodic = level.rolling(4).sum(), level
        growth = pd.DataFrame({"Annual Growth": annual, "Quarterly Growth": periodic})
        growth, units = recalibrate(growth, "Thousands")
        growth_plot_finalise(
            growth,
            plot_from=recent,
            title=title,
            ylabel=f"{units} of persons",
            rfooter=f"{SOURCE_3101} 310101",
            lfooter=f"Australia. {stype} series. ",
            y0=True,
            zero_y=True,
            legend=True,
            pre_tag="erp",
            tag="level-growth",
            show=SHOW,
            file_type=FILE_TYPE,
        )


plot_level_growth()

## Age profiles (3101.0)

In [15]:
# --- Fetch the nine age-by-state tables (3101051..3101059) ---
age_data: dict[str, pd.DataFrame] = {}
_age_meta_parts: list[pd.DataFrame] = []
for _t in AGE_TABLES:
    _d, _m = ra.read_abs_cat("3101.0", single_excel_only=_t, verbose=False)
    age_data[_t] = _d[_t]
    _age_meta_parts.append(_m[_m[mc.table] == _t])
age_meta = pd.concat(_age_meta_parts)

In [16]:
def get_age_data(table: str, group: str) -> tuple[str, pd.DataFrame]:
    """Extract an age-distribution DataFrame and the (abbreviated) state name."""

    relevant = age_meta[
        (age_meta[mc.table] == table) & age_meta[mc.did].str.contains(group)
    ]
    state = relevant["Table Description"].iloc[0].split(",")[-1].strip()
    abbreviation = abbreviate_state(state)

    columns = relevant[mc.id]
    data = age_data[table][columns]
    labels = (
        relevant[mc.did]
        .str.rsplit(";", n=2)
        .str[-2]
        .str.replace("100 and over", "100")
        .astype(int)
    )
    data_i = pd.DataFrame(data.to_numpy(), columns=labels, index=data.index)
    return abbreviation, data_i


def calculate_medians(data: pd.DataFrame) -> pd.Series:
    """Interpolated median age from an age-distribution DataFrame."""

    half = 0.5
    row_total = data.sum(axis=1)
    cumulative = data.div(row_total, axis=0).cumsum(axis=1)
    whole_median_age = cumulative.gt(half).idxmax(axis=1) - 1

    low = pd.Series({
        x: cumulative.loc[x, y]
        for x, y in zip(whole_median_age.index, whole_median_age.values)
    })
    high = pd.Series({
        x: cumulative.loc[x, y + 1]
        for x, y in zip(whole_median_age.index, whole_median_age.values)
    })
    fractional_age = (half - low) / (high - low)
    return whole_median_age + fractional_age

In [17]:
def plot_state_age_profiles() -> None:
    """Population age distribution by jurisdiction, for Female / Male / Persons."""

    for group in GROUPS:
        compositions = {}
        period = None
        for table in AGE_TABLES:
            state, data = get_age_data(table, group)
            period = data.index[-1]
            latest = data.iloc[-1]
            pct = (latest / latest.sum()) * 100
            compositions[state] = hma(pct, HMA_SMOOTHER)

        df = pd.DataFrame(compositions)
        state_colors = [STATE_COLORS[x] for x in df.columns]
        ax = df.plot(lw=2, color=state_colors, **LINESTYLE)
        finalise_plot(
            axes=ax,
            title=f"Population distribution by Age and Jurisdiction ({group})",
            ylabel="Kernel Density Estimate (%)",
            xlabel="Age in whole years",
            legend={"loc": "best", "fontsize": "small", "ncols": 3},
            tag=group,
            lfooter=f"Australia. {period}",
            rfooter=f"Calculated from {SOURCE_3101} {[int(i) for i in AGE_TABLES]}",
            pre_tag="erp",
            show=SHOW,
        )


def plot_median_age_by_state() -> None:
    """Time series of median age by jurisdiction, for Female / Male / Persons."""

    for group in GROUPS:
        medians = {}
        for table in AGE_TABLES:
            state, df = get_age_data(table, group)
            medians[state] = calculate_medians(df)
        data = pd.DataFrame(medians)
        colors = [get_color(x) for x in data.columns]

        line_plot_finalise(
            data,
            color=colors,
            **LINESTYLE,
            title=f"Median Population Age by Jurisdiction ({group})",
            ylabel="Years",
            xlabel=None,
            legend={"loc": "best", "fontsize": "small", "ncols": 3},
            lfooter="Australia. ",
            rfooter=f"Calculated from {SOURCE_3101} {[int(i) for i in AGE_TABLES]}",
            width=2,
            pre_tag="erp",
            show=SHOW,
        )


def plot_age_gender_profiles() -> None:
    """Median age by gender for each state and territory."""

    colors = ["hotpink", "cornflowerblue"]
    for table in AGE_TABLES:
        gender_medians = {}
        state = ""
        for group in GROUPS[0:2]:  # assumes Persons is last
            state, data = get_age_data(table, group)
            gender_medians[group] = calculate_medians(data)
        df = pd.DataFrame(gender_medians)

        line_plot_finalise(
            df,
            color=colors,
            title=f"Median Population Age by Gender for {state}",
            ylabel="Years",
            rfooter=f"Calculated from {SOURCE_3101} {table}",
            width=2,
            pre_tag="erp",
            show=SHOW,
        )


plot_state_age_profiles()
plot_median_age_by_state()
plot_age_gender_profiles()

## Population Growth and Migration

Multi-source measures: ABS (3101.0) ERP vs LFS (6202.0) civilian population 15+ vs an implicit population derived from the National Accounts (5206.0). Net migration proxies combine 3101.0 and 3401.0.

In [18]:
def plot_population_estimates() -> None:
    """Three independent population estimates compared."""

    pop = collect_growth_data()
    use = [
        "Estimated Resident Population",
        "LFS Civilian Population 15+",
        "Implicit population from National Accounts",
    ]
    frame = pd.DataFrame({k: pop[k] for k in use})
    frame, units = recalibrate(frame, "Thousands")

    multi_start(
        frame,
        function=line_plot_finalise,
        starts=(0, -RECENT_MONTHS),
        title="Population Estimates",
        ylabel=units,
        dropna=True,
        width=[2.5, 2, 1.5],
        style=["-", "--", "-."],
        lfooter="Australia. ",
        rfooter=SOURCE_POP,
        pre_tag="multi",
        show=SHOW,
        file_type=FILE_TYPE,
    )


plot_population_estimates()

In [19]:
def plot_population_growth() -> None:
    """Annual absolute growth from three population concepts, plus net migration proxy."""

    pop = collect_growth_data()
    use = [
        "Estimated Resident Population Annual Growth",
        "LFS Civilian Population 15+ Annual Growth",
        "Implicit population (from National Accounts) growth",
        "ERP Growth less Natural Increase",
    ]
    frame = pd.DataFrame({k: pop[k] for k in use})
    frame.rename(
        columns={
            "ERP Growth less Natural Increase":
                "ERP Growth less Natural Increase (Net migration proxy)",
        },
        inplace=True,
    )
    frame, _ = recalibrate(frame, "Thousands")

    multi_start(
        frame,
        function=line_plot_finalise,
        title="Population Growth",
        starts=(0, frame.index[-RECENT_MONTHS]),
        ylabel="Thousands per year",
        dropna=True,
        width=[2.5, 2, 1.5, 1],
        style=["-", "--", ":", "-."],
        color=["brown", "blue", "orange", "cornflowerblue"],
        y0=True,
        annotate=True,
        lfooter="Australia. ERP = Estimated Resident Population. ",
        rfooter=SOURCE_POP,
        pre_tag="multi",
        show=SHOW,
        file_type=FILE_TYPE,
    )


def plot_population_growth_rate() -> None:
    """Annual growth rate (%) across three population concepts."""

    pop = collect_growth_data()
    use = [
        "Estimated Resident Population Annual Growth Rate",
        "LFS Civilian Population 15+ Annual Growth Rate",
        "Implicit population (from National Accounts) growth rate",
    ]
    frame = pd.DataFrame({k: pop[k] for k in use})

    multi_start(
        frame,
        function=line_plot_finalise,
        starts=(0, -RECENT_MONTHS),
        title="Population Growth Rate",
        ylabel="Annual percentage change",
        dropna=True,
        width=[2, 1.5, 1],
        style=["-", "--", "-", ":"],
        y0=True,
        lfooter="Australia. ",
        rfooter=SOURCE_POP,
        pre_tag="multi",
        show=SHOW,
        file_type=FILE_TYPE,
    )


plot_population_growth()
plot_population_growth_rate()

In [20]:
def plot_implied_migration() -> None:
    """ERP growth less natural increase vs 12m net arrivals."""

    pop = collect_growth_data()
    frame = pd.DataFrame({
        "ERP Growth less Natural Increase": pop["ERP Growth less Natural Increase"],
        "12m Rolling Net Arrivals": pop["12 month rolling net total arrivals"],
    })
    frame, units = recalibrate(frame, "Thousands")

    multi_start(
        frame,
        function=line_plot_finalise,
        starts=(0, -RECENT_MONTHS),
        title="Implied Net Migration: ERP Growth less Natural Increase",
        ylabel=f"{units} / year",
        dropna=True,
        width=[2, 1],
        y0=True,
        annotate=True,
        legend=True,
        lfooter=(
            "Australia. Original series. "
            "ERP = Estimated Resident Population. "
            "Natural increase: rolling 4-quarter sum. "
        ),
        rfooter="ABS Cat. 3101.0, 3401.0",
        pre_tag="multi",
        show=SHOW,
        file_type=FILE_TYPE,
    )


def plot_growth_components() -> None:
    """Stacked view of ERP growth components: births, deaths (negative),
    natural increase, net migration, and total ERP growth."""

    pop = collect_growth_data()
    frame = pd.DataFrame({
        "Deaths (negative)": -pop["Annual Deaths"],
        "Births": pop["Annual Births"],
        "Natural Increase": pop["Annual Natural Increase"],
        "Net Migration (ERP growth less natural increase)":
            pop["ERP Growth less Natural Increase"],
        "ERP Annual Growth": pop["Estimated Resident Population Annual Growth"],
    })
    frame, units = recalibrate(frame, "Thousands")

    multi_start(
        frame,
        function=line_plot_finalise,
        starts=(0, -RECENT_MONTHS),
        title="Components of Population Growth",
        ylabel=f"{units} / year",
        dropna=True,
        width=[2, 2, 1, 2, 2],
        color=["black", "brown", "blue", "darkorange", "cornflowerblue"],
        style=["-", "-", ":", "-", "--"],
        y0=True,
        annotate=True,
        legend=True,
        lfooter=(
            "Australia. Original series. Rolling 4-quarter sums. "
            "Natural increase = births - deaths. "
            "ERP = Estimated Resident Population. "
        ),
        rfooter=SOURCE_3101,
        pre_tag="multi",
        show=SHOW,
        file_type=FILE_TYPE,
    )


plot_implied_migration()
plot_growth_components()


### Growth split: natural increase vs net migration

In [21]:
def plot_growth_split() -> None:
    """Two-line split of ERP annual growth: natural increase vs net
    migration (ERP growth less natural increase), charted twice: as
    numbers and as rates (per cent of year-earlier ERP). On the rates
    chart the two lines sum to the published ERP annual growth rate."""

    pop = collect_growth_data()
    ni = pop["Annual Natural Increase"]
    mig = pop["ERP Growth less Natural Increase"]
    erp_year_earlier = pop["Estimated Resident Population"].shift(12)

    common = {
        "dropna": True,
        "width": [2, 2],
        "y0": True,
        "annotate": True,
        "legend": True,
        "rfooter": SOURCE_3101,
        "pre_tag": "multi",
        "show": SHOW,
        "file_type": FILE_TYPE,
    }

    # numbers
    frame = pd.DataFrame({
        "Natural Increase": ni,
        "Net Migration (ERP growth less natural increase)": mig,
    })
    frame, units = recalibrate(frame, "Thousands")
    multi_start(
        frame,
        function=line_plot_finalise,
        starts=(0, -RECENT_MONTHS),
        title="Population Growth: Natural Increase vs Net Migration",
        ylabel=f"{units} / year",
        lfooter=(
            "Australia. Original series. Rolling 4-quarter sums. "
            "Net migration = ERP growth less natural increase. "
            "ERP = Estimated Resident Population. "
        ),
        **common,
    )

    # rates: the two lines sum to the published ERP annual growth rate
    rates = pd.DataFrame({
        "Natural Increase": ni / erp_year_earlier * 100,
        "Net Migration (ERP growth less natural increase)": mig / erp_year_earlier * 100,
    })
    multi_start(
        rates,
        function=line_plot_finalise,
        starts=(0, -RECENT_MONTHS),
        title="Population Growth Rate: Natural Increase vs Net Migration",
        ylabel="Per cent per year",
        lfooter=(
            "Australia. Original series. Rolling 4-quarter sums. "
            "Net migration = ERP growth less natural increase. "
            "Rates relative to ERP a year earlier. "
        ),
        **common,
    )


plot_growth_split()

In [22]:
def _get_nom_monthly() -> Series:
    """Net Overseas Migration from 310101, 4Q rolling sum, upsampled to monthly."""
    stype = "Original"
    selector = {"310101": mc.table, stype: mc.stype, "Net Overseas Migration": mc.did}
    _t, nom_sid, _ = ra.find_abs_id(meta_310101, selector, verbose=False)
    nom_q = df_310101[nom_sid].rolling(4).sum()
    return ra.qtly_to_monthly(nom_q)


def plot_migration_tty() -> None:
    """TTY migrant population growth: NOM vs smoothed net arrivals."""

    pop = collect_growth_data()
    net_arrivals = pop["12 month rolling net total arrivals"].dropna()
    smoothed = hma(net_arrivals, 25)
    nom = _get_nom_monthly()

    frame = pd.DataFrame({
        "Net Overseas Migration (4Q rolling sum)": nom,
        "Net Arrivals Proxy (12m net, 25-term HMA)": smoothed,
    })
    frame, units = recalibrate(frame, "Thousands")

    multi_start(
        frame,
        function=line_plot_finalise,
        starts=(0, -RECENT_MONTHS),
        title="Migrant Population Growth (TTY)",
        ylabel=f"{units} / year",
        dropna=True,
        width=[2, 1],
        style=["-", "--"],
        y0=True,
        annotate=True,
        legend=True,
        lfooter=(
            "Australia. Original series. "
            "NOM: 4-quarter rolling sum. "
            "Net arrivals=arrivals-departures, 12m sum, 25-term HMA. "
        ),
        rfooter="ABS Cat. 3101.0, 3401.0",
        pre_tag="multi",
        show=SHOW,
        file_type=FILE_TYPE,
    )


def _erp_monthly() -> Series:
    """Return monthly ERP (thousands), no forward projection."""

    erp_q, _ = get_population(project=False)
    return ra.qtly_to_monthly(erp_q) / 1_000


def plot_migration_growth_rate() -> None:
    """Migrant growth rate as % of ERP: NOM and net-arrivals proxy."""

    pop = collect_growth_data()
    erp = _erp_monthly()
    smoothed = hma(pop["12 month rolling net total arrivals"].dropna(), 25)
    nom = _get_nom_monthly()

    frame = pd.DataFrame({
        "Net Overseas Migration (4Q rolling sum)": (nom / erp) * 100,
        "Net Arrivals Proxy (12m net, 25-term HMA)": (smoothed / erp) * 100,
    })

    multi_start(
        frame,
        function=line_plot_finalise,
        starts=(0, -RECENT_MONTHS),
        title="Migrant Population Growth Rate (TTY)",
        ylabel="Per cent of ERP per year",
        dropna=True,
        width=[2, 1.5],
        style=["-", "--"],
        y0=True,
        annotate=True,
        legend=True,
        lfooter=(
            "Australia. Original series. ERP=Estimated Resident Population. "
            "NOM: 4-quarter rolling sum. "
            "Net arrivals=arrivals-departures, 12m sum, 25-term HMA. "
        ),
        rfooter="ABS Cat. 3101.0, 3401.0",
        pre_tag="multi",
        show=SHOW,
        file_type=FILE_TYPE,
    )


def plot_migration_share_of_growth() -> None:
    """Migration as a percentage of total population growth (both measures).

    COVID-era values are masked: the ERP-growth denominator became
    tiny/negative through 2020-2022, producing extreme ratios that
    swamp the rest of the series.
    """

    pop = collect_growth_data()
    erp = _erp_monthly()
    erp_growth = erp.diff(12)
    nom = _get_nom_monthly()

    # The net-arrivals share is computed on raw values and then
    # HMA-smoothed in pre- and post-COVID segments independently,
    # so the 25-term window does not bleed COVID into the shoulders.
    covid_start = pd.Period("2020-01", freq="M")
    covid_end = pd.Period("2023-02", freq="M")
    net = pop["12 month rolling net total arrivals"].dropna()
    raw_share = ((net / erp_growth) * 100).dropna()
    pre = raw_share[raw_share.index < covid_start]
    post = raw_share[raw_share.index > covid_end]
    smoothed_share = pd.concat([hma(pre, 25), hma(post, 25)])

    frame = pd.DataFrame({
        "Net Overseas Migration (4Q rolling sum)": (nom / erp_growth) * 100,
        "Net Arrivals Proxy (raw)": raw_share,
        "Net Arrivals Proxy (25-term HMA)": smoothed_share,
    })
    covid_mask = (frame.index >= covid_start) & (frame.index <= covid_end)
    frame.loc[covid_mask] = pd.NA

    multi_start(
        frame,
        function=line_plot_finalise,
        starts=(0, -RECENT_MONTHS),
        title="Migration as a Share of Population Growth",
        ylabel="Per cent of ERP annual growth",
        dropna=False,
        width=[1, 0.75, 2],
        style=["-", "-", "--"],
        color=["darkblue", "darkorange", "darkorange"],
        alpha=[1.0, 0.4, 1.0],
        annotate=[True, False, True],
        y0=True,
        legend=True,
        axvspan={"xmin": covid_start, "xmax": covid_end,
                 "color": "mistyrose", "alpha": 0.6, "label": "COVID (excluded)"},
        lfooter=(
            "Australia. Original series. ERP=Estimated Resident Population. "
            "NOM: 4-quarter rolling sum. "
            "Net arrivals=arrivals-departures, 12m sum, 25-term HMA. "
        ),
        rfooter="ABS Cat. 3101.0, 3401.0",
        pre_tag="multi",
        show=SHOW,
        file_type=FILE_TYPE,
    )


plot_migration_tty()
plot_migration_growth_rate()
plot_migration_share_of_growth()

### Forward NOM proxy

A forward-looking proxy for Net Overseas Migration built from the monthly LFS civilian population aged 15+ (6202.0) - the timely lead - converted to migration with ERP demographic components (3101.0: ageing-in, 15+ deaths, child migration). It runs about two quarters ahead of the official NOM release.

In [23]:
def _build_nom_forward_proxy() -> tuple[Series, Series, pd.Period]:
    """Build the 6202-based forward proxy for Net Overseas Migration.

    proxy = civ15 year-on-year growth (6202, monthly - the timely lead)
            - 15-year-old ageing-in  +  15+ deaths  +  child migration
          = civ15 growth - 15+ natural increase + child migration

    The +deaths term is essential, not optional: in a headcount change a death
    is indistinguishable from an emigration, so without crediting deaths back
    they are mistaken for people leaving (it sits bundled with ageing-in as the
    ~flat 150k 15+ natural increase). Child migration (0-14 ERP cohort survival,
    3101.0) is observed while ERP-by-age exists, then extended over the forward
    quarters by a ratio of growth derived from the data (not hardcoded), so it
    scales with the migration cycle.

    Returns (proxy, official_nom, last_official_quarter); quarterly TTY, '000.
    """
    # timely engine: civ15 (6202) year-on-year growth
    civ15, _ = get_population("civ15", state="Australia")
    growth = civ15.resample("Q").mean().diff(4)

    # single-year-of-age ERP (3101.0): ageing-in (15-year-olds) and the 0-14 pyramid
    def age_ge(a: int) -> Series:
        return _erp_age_sum(a) / 1_000.0                       # persons -> '000, annual June
    n15_yr = age_ge(15) - age_ge(16)                           # count of 15-year-olds
    childmig_yr = ((age_ge(1) - age_ge(16))
                   - (age_ge(0) - age_ge(15)).shift(1)).dropna()
    n15 = _interp_june(n15_yr, growth.index)
    childmig_obs = _interp_june(childmig_yr, growth.index)

    # NOM and 15+ deaths from 3101.0 (4-quarter rolling)
    def nom_component(did: str) -> Series:
        sel = {"310101": mc.table, "Original": mc.stype, did: mc.did}
        _t, sid, _ = ra.find_abs_id(meta_310101, sel, verbose=False)
        return df_310101[sid].rolling(4).sum()
    deaths = nom_component("Deaths ;  Australia ;")
    nom = nom_component("Net Overseas Migration ;  Australia ;").dropna()

    natural_increase = (n15 - deaths).ffill()                  # ageing-in less deaths (~flat 150)
    m15 = growth - natural_increase                            # = growth - ageing-in + deaths

    # extend child migration over forward quarters by a derived ratio of growth
    last_age = pd.Period(f"{childmig_yr.index[-1].year}Q2", freq="Q-DEC")
    ratio = (childmig_obs.loc[:last_age] / growth.loc[:last_age]).dropna().median()
    childmig = childmig_obs.copy()
    childmig[childmig.index > last_age] = ratio * growth[childmig.index > last_age]

    proxy = (m15 + childmig).dropna()
    return proxy, nom, nom.index[-1]


def plot_nom_forward_proxy() -> None:
    """6202 forward NOM proxy vs official NOM (full history + recent)."""
    proxy, nom, last_official = _build_nom_forward_proxy()
    frame = pd.DataFrame({
        "Official NOM (3101.0)": nom,
        "6202 Forward Proxy": proxy,
    })
    multi_start(
        frame,
        function=line_plot_finalise,
        starts=(0, -(RECENT_MONTHS // 3)),                     # full + ~recent (quarters)
        width=[2, 1.5],
        style=["-", "--"],
        annotate=True,
        rounding=0,
        dropna=True,
        title="Net Overseas Migration: 6202 Forward Proxy",
        ylabel="Persons per year ('000)",
        axvline={"x": last_official, "color": "grey", "linestyle": ":", "linewidth": 1},
        rheader="Proxy = civ15 growth - 15yo ageing-in + 15+ deaths + child migration",
        lfooter=(
            "Australia. Dotted line: last official NOM. "
            "Forward child migration held at its recent share of growth. "
        ),
        rfooter="ABS 6202.0, 3101.0",
        pre_tag="multi",
        show=SHOW,
        file_type=FILE_TYPE,
    )


plot_nom_forward_proxy()

In [24]:
def plot_nplt_vs_nom() -> None:
    """Compare Net Permanent and Long-term Arrivals (NPLT) with Net Overseas Migration."""

    # NPLT: 12m rolling net Permanent and Long-term
    stype = "Original"
    plt_did = "Number of movements ;  Permanent and Long-term Arrivals ;"
    dep_did = "Number of movements ;  Permanent and Long-term Departures ;"
    _t, arr_sid, _ = ra.find_abs_id(
        meta_3401, {plt_did: mc.did, "340101": mc.table, stype: mc.stype},
    )
    _t, dep_sid, _ = ra.find_abs_id(
        meta_3401, {dep_did: mc.did, "340102": mc.table, stype: mc.stype},
    )
    net_plt = (abs_3401["340101"][arr_sid] - abs_3401["340102"][dep_sid])
    net_plt_12m = net_plt.rolling(12).sum() / 1_000

    # Net Overseas Migration from 3101.0 (quarterly, 4Q rolling sum)
    selector = {"310101": mc.table, stype: mc.stype, "Net Overseas Migration": mc.did}
    _t, nom_sid, _ = ra.find_abs_id(meta_310101, selector, verbose=False)
    nom_q = df_310101[nom_sid].rolling(4).sum()
    nom, _ = recalibrate(ra.qtly_to_monthly(nom_q), "Persons")

    frame = pd.DataFrame({
        "Net Permanent and Long-term Arrivals (12m rolling)": net_plt_12m,
        "Net Overseas Migration (4Q rolling sum)": nom,
    })
    frame, units = recalibrate(frame, "Thousands")

    multi_start(
        frame,
        function=line_plot_finalise,
        starts=(0, -RECENT_MONTHS),
        title="Net Permanent and Long-term Arrivals vs Net Overseas Migration",
        ylabel=f"{units} / year",
        rheader="The ABS cautions against using NPLT as a migration proxy.",
        dropna=True,
        width=[1.5, 2],
        style=["-.", "-"],
        y0=True,
        annotate=True,
        legend=True,
        lfooter=(
            "Australia. Original series. "
            "Net PLT: 12m rolling sum. "
            "NOM: 4-quarter rolling sum. "
        ),
        rfooter="ABS Cat. 3101.0, 3401.0",
        pre_tag="multi",
        show=SHOW,
        file_type=FILE_TYPE,
    )


plot_nplt_vs_nom()

recalibrate(): Units not appropriately calibrated: Persons


### NOM vs ERP growth residual (implied intercensal discrepancy)

Full-history comparison of Net Overseas Migration (NOM, from 3101.0) with the implied net-migration proxy ERP growth less natural increase (ERP-NI). The residual `(ERP-NI) - NOM` is the implied intercensal discrepancy: ABS back-smears the gap between each census count and the running ERP estimate across the preceding ~20 quarters, producing peaks/troughs near each census year. The current open intercensal period (post-2021-Q3) is set to zero by convention and shaded accordingly.


In [25]:
def plot_nom_vs_erp_growth_less_ni() -> None:
    """Full-history NOM vs ERP-NI (net migration proxy)."""

    pop = collect_growth_data()
    erp_ni = pop["ERP Growth less Natural Increase"]
    nom = _get_nom_monthly()

    frame = pd.DataFrame({
        "Net Overseas Migration (4Q rolling sum)": nom,
        "ERP Growth less Natural Increase": erp_ni,
    }).dropna(how="all")
    # Both source series (ERP change, NI, NOM) share the ABS '000' scale.
    erp_growth_unit = pop["Estimated Resident Population Annual Growth"].attrs.get("unit", "")
    frame, units = recalibrate(frame, to_scale_word(erp_growth_unit))

    line_plot_finalise(
        frame,
        title="Net Overseas Migration vs ERP Growth less Natural Increase",
        ylabel=f"{units} of persons / year",
        dropna=True,
        width=[2.5, 1.5],
        style=["-", "--"],
        y0=True,
        annotate=True,
        legend=True,
        lfooter=(
            "Australia. Original series. "
            "NOM: 4-quarter rolling sum. "
            "ERP = Estimated Resident Population. "
        ),
        rfooter=SOURCE_POP,
        pre_tag="multi",
        show=SHOW,
        file_type=FILE_TYPE,
    )


def plot_erp_growth_less_ni_minus_nom() -> None:
    """Full-history (ERP-NI) minus NOM: the implied intercensal discrepancy.

    After each census, ABS back-smears the census-vs-running-estimate gap across
    the preceding ~20 quarters, producing the visible peaks/troughs near each
    census date. The open intercensal period after the most recent census is
    published with the discrepancy set to zero by convention; shaded here to
    make clear the flat tail is arithmetic, not evidence the two measures agree.
    """

    pop = collect_growth_data()
    erp_ni = pop["ERP Growth less Natural Increase"]
    nom = _get_nom_monthly()

    diff = (erp_ni - nom).dropna()
    diff.name = "Implied intercensal discrepancy"
    # ERP change, NI and NOM are all published by ABS in '000' units.
    erp_growth_unit = pop["Estimated Resident Population Annual Growth"].attrs.get("unit", "")
    diff, units = recalibrate(diff, to_scale_word(erp_growth_unit))

    last_census = pd.Period("2021-09", freq="M")
    census_months = [
        pd.Period(f"{y}-09", freq="M")
        for y in (1991, 1996, 2001, 2006, 2011, 2016, 2021)
    ]

    line_plot_finalise(
        diff,
        title="Implied Intercensal Discrepancy: (ERP growth - NI) - NOM",
        ylabel=f"{units} of persons / year",
        dropna=True,
        width=2,
        y0=True,
        annotate=True,
        axvline=[
            {"x": c, "color": "lightgrey", "linestyle": ":", "linewidth": 0.8}
            for c in census_months
        ],
        axvspan={
            "xmin": last_census, "xmax": diff.index[-1],
            "color": "lightgrey", "alpha": 0.35,
            "label": "Open intercensal period (discrepancy = 0 by convention)",
        },
        legend=True,
        lfooter=(
            "Australia. Original series. 4-quarter rolling sums. "
            "Peaks/troughs near census years reflect ABS back-smearing each "
            "census revision across the prior 20 quarters. "
            "Shaded tail: open intercensal period, zero by convention until the "
            "2026 census is processed. "
        ),
        rfooter=SOURCE_POP,
        pre_tag="multi",
        show=SHOW,
        file_type=FILE_TYPE,
    )


plot_nom_vs_erp_growth_less_ni()
plot_erp_growth_less_ni_minus_nom()

## Finished

In [26]:
# watermark
%load_ext watermark
%watermark -u -t -d --iversions --watermark --machine --python --conda

Last updated: 2026-06-20 09:33:15

Python implementation: CPython
Python version       : 3.14.2
IPython version      : 9.14.1

conda environment: n/a

Compiler    : Clang 21.1.4 
OS          : Darwin
Release     : 25.5.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

mgplot     : 0.2.27
pandas     : 3.0.3
re         : 2.2.1
readabs    : 0.2.3
statsmodels: 0.14.6
typing     : 3.10.0.0

Watermark: 2.6.0



In [27]:
print("Finished")

Finished
